# smolagents Teaching Tutorial Notebook
## 10 Agent Applications from Basic to Intermediate

This notebook teaches **Hugging Face `smolagents`** by building 10 small applications.

The learning path starts with a simple no-tool agent, then moves into custom tools, web search, Hugging Face Hub exploration, dataset exploration, CSV analysis, local RAG, tool-calling dispatch, and a small supervisor-style multi-agent system.

### What you will build

| # | Application | Level | Main idea |
|---|---|---|---|
| 1 | First CodeAgent | Basic | Run a simple agent with no custom tools |
| 2 | Calculator Tutor | Basic | Add deterministic math tools |
| 3 | Text Coach | Basic | Build a writing helper with tools |
| 4 | Web Research Assistant | Basic+ | Use a web-search tool |
| 5 | Hugging Face Model Finder | Basic+ | Query the Hugging Face Hub |
| 6 | Dataset Explorer | Intermediate- | Inspect datasets from the Hub |
| 7 | CSV Data Analyst | Intermediate | Analyze a local CSV file |
| 8 | Course Notes RAG Agent | Intermediate | Retrieve local knowledge before answering |
| 9 | Support Ticket Dispatcher | Intermediate | Use `ToolCallingAgent` for structured tool calls |
| 10 | Supervisor Multi-Agent System | Intermediate | Route work to specialist agents |

### Do you need a Hugging Face certificate?

No certificate is required to use `smolagents` or run this notebook.  
You may need a **free Hugging Face account and access token** to call hosted models through the Hugging Face Inference API or inference providers.  
If your goal is a credential, use the final section, **Hugging Face Certification Helper**, as a checklist for the Hugging Face Agents Course.

## 0. Setup

Run this cell first. It installs the main packages used in this tutorial.

> In Google Colab, this may take a minute. After installing, restart the runtime if Python asks you to.

In [ ]:
# Core install for this notebook
# - smolagents[toolkit] includes common tools such as web search support
# - huggingface_hub gives access to models and Hub metadata
# - datasets supports public dataset exploration
# - pandas/numpy/sklearn are used in the data and RAG examples

%pip install -U "smolagents[toolkit]" huggingface_hub datasets pandas numpy scikit-learn gradio -q

## 1. Hugging Face Token Setup

Most examples use `InferenceClientModel`, which calls a hosted model through Hugging Face. If you already have `HF_TOKEN` set in your environment, the notebook will use it. Otherwise, paste a token when prompted.

**Recommended token scope:** read access is enough for these examples.

You can still read and study the notebook without a token, but model calls may fail until authentication is configured.

In [ ]:
import os
from getpass import getpass

# Optional: comment this block out if your environment already has HF_TOKEN.
if not os.environ.get("HF_TOKEN"):
    token = getpass("Paste your Hugging Face token, or press Enter to skip for now: ").strip()
    if token:
        os.environ["HF_TOKEN"] = token
        print("HF_TOKEN has been set for this notebook session.")
    else:
        print("No token set. You can still review the notebook, but hosted model calls may fail.")
else:
    print("HF_TOKEN is already available in this environment.")

## 2. Common Imports and Model Helper

`smolagents` supports multiple model backends. This notebook defaults to `InferenceClientModel`.

You can change `DEFAULT_MODEL_ID` to another instruction-tuned model available through Hugging Face inference providers.

In [ ]:
from smolagents import CodeAgent, ToolCallingAgent, InferenceClientModel, Tool, tool

# Search tool names can vary by smolagents version. This fallback keeps the notebook friendly.
try:
    from smolagents import DuckDuckGoSearchTool as SearchTool
    SEARCH_TOOL_NAME = "DuckDuckGoSearchTool"
except Exception:
    from smolagents import WebSearchTool as SearchTool
    SEARCH_TOOL_NAME = "WebSearchTool"

DEFAULT_MODEL_ID = None  # None lets smolagents use its default hosted model.
# Example alternative:
# DEFAULT_MODEL_ID = "Qwen/Qwen2.5-Coder-32B-Instruct"


def get_model(model_id: str | None = DEFAULT_MODEL_ID):
    """Create a Hugging Face inference-backed model for smolagents."""
    kwargs = {}
    if model_id:
        kwargs["model_id"] = model_id
    if os.environ.get("HF_TOKEN"):
        kwargs["token"] = os.environ["HF_TOKEN"]
    return InferenceClientModel(**kwargs)

model = get_model()
print(f"Ready. Search tool selected: {SEARCH_TOOL_NAME}")

# Application 1 — First CodeAgent

### Goal
Create the smallest useful `CodeAgent` and ask it to teach a concept.

### Teaching point
A `CodeAgent` can solve tasks by generating and executing Python actions. In this first example, we do not add custom tools yet.

In [ ]:
first_agent = CodeAgent(
    tools=[],
    model=model,
    add_base_tools=True,
)

result = first_agent.run(
    "Explain AI agents to a high-school student in 5 bullets. "
    "Use one simple analogy and one warning about when not to use agents."
)

print(result)

### Reflection

This first application shows the basic pattern:

1. Choose a model.
2. Create an agent.
3. Give the agent a task.
4. Print the result.

This is useful for concept explanation, planning, light reasoning, and tutorial generation.

# Application 2 — Calculator Tutor with Custom Tools

### Goal
Add deterministic tools for math operations.

### Teaching point
Agents are strongest when they can call reliable tools instead of guessing. Here, the LLM decides when to use a tool, while the tool performs exact computation.

In [ ]:
@tool
def percent_change(old_value: float, new_value: float) -> str:
    """
    Calculate the percentage change from an old value to a new value.

    Args:
        old_value: The original value.
        new_value: The new value.
    """
    if old_value == 0:
        return "Cannot calculate percent change when old_value is 0."
    change = ((new_value - old_value) / old_value) * 100
    direction = "increase" if change >= 0 else "decrease"
    return f"{abs(change):.2f}% {direction}"


@tool
def grade_needed(current_average: float, target_average: float, final_weight_percent: float) -> str:
    """
    Calculate the final exam grade needed to reach a target course average.

    Args:
        current_average: Current average before the final exam, from 0 to 100.
        target_average: Desired final course average, from 0 to 100.
        final_weight_percent: Final exam weight as a percent of the total grade.
    """
    w = final_weight_percent / 100
    if not 0 < w <= 1:
        return "final_weight_percent must be between 0 and 100."
    needed = (target_average - current_average * (1 - w)) / w
    return f"You need {needed:.2f}% on the final exam."


calculator_tutor = CodeAgent(
    tools=[percent_change, grade_needed],
    model=model,
    add_base_tools=True,
)

answer = calculator_tutor.run(
    "My current average is 88. The final is worth 25%. "
    "What final exam grade do I need to finish with an 80? "
    "Also explain the calculation briefly."
)
print(answer)

### Try it
Change the numbers in the prompt and rerun the cell. Notice that the agent can explain the result, but the math comes from your deterministic tool.

# Application 3 — Text Coach Agent

### Goal
Create tools that help an agent clean and improve writing.

### Teaching point
Tools do not have to be external APIs. Many useful agent tools are small local functions.

In [ ]:
@tool
def clean_spacing(text: str) -> str:
    """
    Clean repeated whitespace and common spacing issues in text.

    Args:
        text: Text to clean.
    """
    import re
    text = re.sub(r"\s+", " ", text).strip()
    text = text.replace(" .", ".").replace(" ,", ",").replace(" ;", ";").replace(" :", ":")
    return text


@tool
def sentence_count(text: str) -> str:
    """
    Count approximate sentences in text.

    Args:
        text: Text to analyze.
    """
    import re
    sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    return f"Approximate sentence count: {len(sentences)}"


text_coach = CodeAgent(
    tools=[clean_spacing, sentence_count],
    model=model,
    add_base_tools=True,
)

messy_text = """
The class went well.   Its just I could not get to week 4 , because the materials only went up to week 3 .
I had my old notes, which covered week 4, because I had done this class before.
"""

response = text_coach.run(
    f"Clean this message, fix grammar, and keep it professional but natural. "
    f"Then give me a one-sentence explanation of what changed. Text: {messy_text}"
)
print(response)

### Teaching move
This is a good pattern for classroom notebooks: make a simple tool, then ask the agent to use it in a realistic writing task.

# Application 4 — Web Research Assistant

### Goal
Give the agent a search tool.

### Teaching point
Search tools let an agent fetch current information. This is useful when the answer may change over time.

> Web search depends on your environment and network access. If the search tool fails, rerun later or use another model/provider.

In [ ]:
search_agent = CodeAgent(
    tools=[SearchTool()],
    model=model,
    add_base_tools=True,
)

research_answer = search_agent.run(
    "Research the current Hugging Face smolagents library at a high level. "
    "Return 5 bullets: purpose, main agent classes, model options, tool options, and one safety note."
)
print(research_answer)

### Extension
Ask the research agent to compare `smolagents`, LangGraph, and LlamaIndex at a high level.

# Application 5 — Hugging Face Model Finder

### Goal
Build a tool that searches the Hugging Face Hub for popular models by task.

### Teaching point
A custom tool can wrap the Hugging Face Hub API and give the agent structured model discovery powers.

In [ ]:
@tool
def top_hf_models_for_task(task: str, limit: int = 5) -> str:
    """
    Return top downloaded Hugging Face models for a task.

    Args:
        task: Hugging Face task name, such as text-classification, text-generation, image-classification, or automatic-speech-recognition.
        limit: Number of models to return.
    """
    from huggingface_hub import list_models

    rows = []
    for model_info in list_models(filter=task, sort="downloads", direction=-1, limit=limit):
        rows.append(f"- {model_info.id} | downloads: {getattr(model_info, 'downloads', 'unknown')}")

    if not rows:
        return f"No models found for task: {task}"
    return "\n".join(rows)


model_finder_agent = CodeAgent(
    tools=[top_hf_models_for_task],
    model=model,
    add_base_tools=True,
)

model_answer = model_finder_agent.run(
    "Find 5 popular Hugging Face models for text-classification. "
    "Then recommend one beginner-friendly use case for the class."
)
print(model_answer)

### Try it
Change `text-classification` to:

- `text-generation`
- `image-classification`
- `automatic-speech-recognition`
- `sentence-similarity`

# Application 6 — Dataset Explorer Agent

### Goal
Create an agent that can inspect public datasets on the Hugging Face Hub.

### Teaching point
Agents become more useful when tools return metadata the LLM can summarize, compare, and teach.

In [ ]:
@tool
def inspect_hf_dataset(dataset_id: str) -> str:
    """
    Inspect a public Hugging Face dataset and return a compact summary.

    Args:
        dataset_id: Dataset repository ID, such as ag_news, glue, or imdb.
    """
    from datasets import get_dataset_config_names, load_dataset_builder

    try:
        configs = get_dataset_config_names(dataset_id)
    except Exception:
        configs = []

    config = configs[0] if configs else None
    builder = load_dataset_builder(dataset_id, name=config) if config else load_dataset_builder(dataset_id)
    info = builder.info

    description = (info.description or "No description available.").strip().replace("\n", " ")[:700]
    features = info.features
    splits = list(info.splits.keys()) if info.splits else []

    return (
        f"Dataset: {dataset_id}\n"
        f"Config used: {config}\n"
        f"Splits: {splits}\n"
        f"Features: {features}\n"
        f"Description excerpt: {description}"
    )


dataset_agent = CodeAgent(
    tools=[inspect_hf_dataset],
    model=model,
    add_base_tools=True,
)

dataset_answer = dataset_agent.run(
    "Inspect the ag_news dataset. Explain what it contains and give me one classroom exercise idea."
)
print(dataset_answer)

### Teaching idea
Have students pick one dataset and answer:

1. What is the prediction target?
2. What are the input columns?
3. What model type could solve it?
4. What ethical or quality concern should we check?

# Application 7 — CSV Data Analyst Agent

### Goal
Analyze a local CSV file with a custom tool.

### Teaching point
This pattern is the bridge from toy agents to practical business applications: the agent explains, but the tool computes.

In [ ]:
import pandas as pd
from pathlib import Path

sales_path = Path("sample_sales.csv")

sales_df = pd.DataFrame(
    [
        {"region": "North", "product": "AI Course", "revenue": 12000, "cost": 5000},
        {"region": "North", "product": "Cloud Course", "revenue": 9000, "cost": 4300},
        {"region": "South", "product": "AI Course", "revenue": 15000, "cost": 6200},
        {"region": "South", "product": "DevOps Course", "revenue": 11000, "cost": 5100},
        {"region": "West", "product": "Cloud Course", "revenue": 7000, "cost": 3500},
        {"region": "West", "product": "DevOps Course", "revenue": 13000, "cost": 5600},
    ]
)
sales_df.to_csv(sales_path, index=False)
sales_df

In [ ]:
@tool
def summarize_sales_csv(csv_path: str) -> str:
    """
    Summarize a sales CSV file with columns region, product, revenue, and cost.

    Args:
        csv_path: Path to the CSV file.
    """
    import pandas as pd

    df = pd.read_csv(csv_path)
    required = {"region", "product", "revenue", "cost"}
    missing = required - set(df.columns)
    if missing:
        return f"Missing required columns: {sorted(missing)}"

    df["profit"] = df["revenue"] - df["cost"]
    df["margin"] = df["profit"] / df["revenue"]

    total_revenue = df["revenue"].sum()
    total_profit = df["profit"].sum()
    by_region = df.groupby("region")[["revenue", "profit"]].sum().sort_values("profit", ascending=False)
    best_product = df.groupby("product")[["revenue", "profit"]].sum().sort_values("profit", ascending=False).head(1)

    return (
        f"Total revenue: ${total_revenue:,.0f}\n"
        f"Total profit: ${total_profit:,.0f}\n\n"
        f"Profit by region:\n{by_region.to_string()}\n\n"
        f"Best product by profit:\n{best_product.to_string()}"
    )


data_analyst_agent = CodeAgent(
    tools=[summarize_sales_csv],
    model=model,
    add_base_tools=True,
)

data_answer = data_analyst_agent.run(
    f"Analyze {sales_path}. Give the executive summary in 5 bullets and include one business recommendation."
)
print(data_answer)

### Why this matters
For real business analytics, avoid letting the LLM invent numbers. Use tools for computation and the agent for explanation, interpretation, and next-step recommendations.

# Application 8 — Local Course Notes RAG Agent

### Goal
Give the agent a small local knowledge base and a retrieval tool.

### Teaching point
RAG means **retrieve first, answer second**. This reduces hallucination because the agent has a grounded source to consult.

In [ ]:
COURSE_NOTES = [
    {
        "title": "Agent Basics",
        "text": "An AI agent uses a model to decide actions. The common loop is think, act, observe, and answer."
    },
    {
        "title": "Tools",
        "text": "A tool is a function or API the agent can call. Tools should have clear names, descriptions, inputs, and outputs."
    },
    {
        "title": "CodeAgent",
        "text": "A CodeAgent writes Python actions. It is expressive and useful for chaining, loops, transformations, and multi-step reasoning."
    },
    {
        "title": "ToolCallingAgent",
        "text": "A ToolCallingAgent emits structured tool calls. It is useful when reliability, validation, and simple API dispatch are important."
    },
    {
        "title": "Safety",
        "text": "Agent systems should minimize unnecessary agency, use deterministic tools where possible, log tool errors, and prefer sandboxed execution for code."
    },
]

@tool
def retrieve_course_notes(query: str, k: int = 3) -> str:
    """
    Retrieve the most relevant local course notes for a query.

    Args:
        query: Student question or topic.
        k: Number of notes to retrieve.
    """
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    texts = [note["title"] + ": " + note["text"] for note in COURSE_NOTES]
    vectorizer = TfidfVectorizer().fit(texts + [query])
    doc_vectors = vectorizer.transform(texts)
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, doc_vectors)[0]
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:k]

    results = []
    for idx, score in ranked:
        note = COURSE_NOTES[idx]
        results.append(f"Title: {note['title']} | Score: {score:.3f}\n{note['text']}")
    return "\n\n".join(results)


rag_agent = CodeAgent(
    tools=[retrieve_course_notes],
    model=model,
    add_base_tools=True,
)

rag_answer = rag_agent.run(
    "Using the course notes, explain the difference between CodeAgent and ToolCallingAgent. "
    "Then give one example use case for each."
)
print(rag_answer)

### Upgrade path
For larger RAG systems, replace the TF-IDF retriever with embeddings and a vector database such as FAISS, Chroma, Qdrant, Pinecone, or Azure AI Search.

# Application 9 — Support Ticket Dispatcher with ToolCallingAgent

### Goal
Use `ToolCallingAgent` for a structured support workflow.

### Teaching point
`ToolCallingAgent` is a good fit when you want the agent to call clear, atomic tools rather than execute open-ended Python code.

This example uses a wireless/core operations theme, but you can adapt it to help desk, cloud operations, DevOps, or healthcare intake.

In [ ]:
@tool
def classify_support_ticket(ticket_text: str) -> str:
    """
    Classify a telecom support ticket into a likely operational domain.

    Args:
        ticket_text: The support ticket text.
    """
    text = ticket_text.lower()
    if any(word in text for word in ["e911", "emergency", "location lookup", "psap"]):
        return "E911 / Emergency Services"
    if any(word in text for word in ["attach", "packet core", "pdu", "session", "apn"]):
        return "Packet Core"
    if any(word in text for word in ["sms", "mms", "messaging"]):
        return "Messaging"
    if any(word in text for word in ["volte", "voice", "ims", "call"]):
        return "Voice / IMS"
    if any(word in text for word in ["auth", "subscriber", "sim", "hss", "udm"]):
        return "Subscriber Identity / Authentication"
    return "General NOC Triage"


@tool
def escalation_path(domain: str, severity: str) -> str:
    """
    Recommend an escalation path for a classified support domain and severity.

    Args:
        domain: Operational domain classification.
        severity: Severity such as low, medium, high, or critical.
    """
    severity = severity.lower().strip()
    compliance = ""
    if "E911" in domain or "Emergency" in domain:
        compliance = " Include compliance owner immediately because emergency service impact may have regulatory risk."
    if severity in ["critical", "high"]:
        return f"Escalate to {domain} SME, NOC incident commander, vendor support if alarms persist, and customer communications lead.{compliance}"
    return f"Route to {domain} queue, collect logs/alarms, monitor trend, and escalate if impact grows.{compliance}"


dispatch_agent = ToolCallingAgent(
    tools=[classify_support_ticket, escalation_path],
    model=model,
)

ticket = "We received alarms for E911 location lookup failures in one market. Several test calls cannot validate location."

dispatch_answer = dispatch_agent.run(
    f"Classify this ticket and recommend the escalation path. Severity is critical. Ticket: {ticket}"
)
print(dispatch_answer)

### Why this is intermediate
The agent has to choose the right sequence:

1. Classify the ticket.
2. Use the classification as input to escalation.
3. Summarize the result clearly.

# Application 10 — Supervisor Multi-Agent System

### Goal
Create a small supervisor-style system with specialist agents.

### Teaching point
A manager agent can coordinate specialized agents. This pattern is useful when one agent should not own every skill.

In this example:

- A **web research agent** handles current web lookup.
- A **data analysis agent** handles CSV sales analysis.
- The **manager agent** coordinates the answer and can also retrieve course notes.

In [ ]:
# Specialist 1: Web research agent
web_research_agent = CodeAgent(
    tools=[SearchTool()],
    model=model,
    add_base_tools=True,
    name="web_research_agent",
    description="Searches the web for current public information. Give it a clear research question."
)

# Specialist 2: Data analysis agent
sales_data_agent = CodeAgent(
    tools=[summarize_sales_csv],
    model=model,
    add_base_tools=True,
    name="sales_data_agent",
    description="Analyzes local sales CSV files. Give it a CSV path and a business question."
)

# Manager / supervisor agent
manager_agent = CodeAgent(
    tools=[retrieve_course_notes],
    model=model,
    add_base_tools=True,
    managed_agents=[web_research_agent, sales_data_agent],
)

manager_prompt = f"""
You are a supervisor agent. Build a short training recommendation for a class learning smolagents.
Use the local course notes to explain the concept.
Use the sales_data_agent to analyze {sales_path}.
Return:
1. The best product opportunity from the CSV.
2. The agent concept this supports.
3. A 3-step classroom activity.
"""

manager_answer = manager_agent.run(manager_prompt)
print(manager_answer)

### Discussion
This is the same architectural idea as a supervisor model in LangGraph, but in a lightweight `smolagents` style:

- Keep specialist agents narrow.
- Give each agent a clear name and description.
- Give the manager agent enough context to decide who should do what.
- Avoid giving every tool to every agent.

# Optional — Gradio Chat UI

### Goal
Launch a simple interface to interact with an agent.

This is useful when turning your tutorial into a classroom demo.

> In some notebook environments, `share=True` may be needed. Use it only when you intentionally want a public temporary link.

In [ ]:
# Optional UI demo.
# Uncomment to launch.

# from smolagents import GradioUI
# GradioUI(manager_agent).launch()

# Hugging Face Certification Helper

No certificate is required to run this notebook. If you want a Hugging Face learning credential, this notebook aligns well with the Hugging Face Agents Course.

## Suggested certification path

### Step 1 — Hugging Face account
Create a Hugging Face account and token. You need this for hosted inference, gated models, pushing projects, and course submissions.

### Step 2 — Agents fundamentals
Study the agents fundamentals:

- What is an agent?
- What are tools?
- What are thoughts, actions, and observations?
- What is the Think → Act → Observe loop?

### Step 3 — smolagents framework
Use this notebook to practice:

- `CodeAgent`
- `ToolCallingAgent`
- `InferenceClientModel`
- custom tools with `@tool`
- custom tools with `Tool`
- RAG-style retrieval
- multi-agent supervision

### Step 4 — Build one portfolio app
Pick one of the 10 applications and expand it into a small portfolio project. Best candidates:

1. CSV Data Analyst Agent
2. Local Course Notes RAG Agent
3. Support Ticket Dispatcher
4. Supervisor Multi-Agent System

### Step 5 — Publish or document your project
For a stronger portfolio, create a Hugging Face Space or GitHub repository with:

- `README.md`
- notebook
- requirements file
- screenshots
- sample prompts
- safety notes

### Step 6 — Prepare for quizzes and final project
Use these review questions:

1. When should you use a deterministic tool instead of agent reasoning?
2. What is the difference between `CodeAgent` and `ToolCallingAgent`?
3. Why should tools have clear inputs, outputs, and descriptions?
4. What are the risks of open-ended code execution?
5. How does RAG reduce hallucination risk?
6. Why is a supervisor agent useful?
7. What should be logged inside tool execution?
8. Why can too many tools make an agent weaker?
9. What information should be included in a project README?
10. How would you evaluate whether an agent succeeded?

# Capstone Challenge

Choose one of these capstones and build it from the notebook patterns.

## Option A — AI Course Assistant
Build an agent that answers questions from course notes, recommends exercises, and generates a short quiz.

Recommended tools:

- `retrieve_course_notes`
- `sentence_count`
- custom quiz generator tool

## Option B — Wireless NOC Assistant
Build an agent that classifies NOC tickets, recommends validation steps, and escalates based on severity.

Recommended tools:

- `classify_support_ticket`
- `escalation_path`
- custom alarm checklist tool

## Option C — Business Data Analyst
Build an agent that reads a CSV, summarizes trends, and writes a management recommendation.

Recommended tools:

- `summarize_sales_csv`
- custom chart summary tool
- optional Gradio UI

## Option D — Hugging Face Discovery Agent
Build an agent that recommends models and datasets for a learning objective.

Recommended tools:

- `top_hf_models_for_task`
- `inspect_hf_dataset`
- optional web search

# Final Review

You now built 10 applications that cover the most important beginner-to-intermediate `smolagents` skills:

- basic agents
- deterministic custom tools
- web search tools
- Hugging Face Hub tools
- dataset tools
- CSV analysis tools
- local RAG
- structured tool calling
- supervisor-style multi-agent systems
- optional UI deployment

The main design rule is simple:

> Let tools compute, retrieve, and validate. Let the agent decide, explain, and coordinate.